# Kopi IoT Web3 — Deep Learning Forecasting (Keras)

Notebook ini mengajarkan **ANN → DNN → LSTM** untuk forecasting deret waktu, lalu
membandingkannya dengan **Naive, Regresi Linear, dan Prophet**.

Alur:
1. Ambil & bersihkan data (ThingSpeak)
2. **Windowing** (deret → data terawasi: 7 hari lalu → prediksi hari ke-8)
3. Latih **MLP (ANN)**, **DNN**, **LSTM** + tampilkan **kurva Loss & MAE per epoch**
4. Bandingkan semua model (MAE / RMSE / Akurasi)

> Catatan jujur: data Anda masih sedikit (~60 hari), jadi deep learning **belum tentu menang**
> akurasi vs model sederhana. Itu pelajaran penting (overfitting & "lapar data"-nya DL).
> Kurva loss/MAE per epoch di sini **otentik** — itulah keunggulan mengajar pakai neural network.

Jalankan: Runtime -> Run all.

In [ ]:
!pip -q install prophet

## 1. Ambil & bersihkan data

In [ ]:
import requests, pandas as pd, numpy as np

CHANNEL_ID = 1848413
RESULTS = 8000
TZ = 'Asia/Jakarta'

url = f'https://api.thingspeak.com/channels/{CHANNEL_ID}/feeds.json'
r = requests.get(url, params={'results': RESULTS, 'timezone': TZ}, timeout=30)
feeds = r.json()['feeds']

df = pd.DataFrame(feeds)
df['created_at'] = pd.to_datetime(df['created_at'])
if df['created_at'].dt.tz is not None:
    df['created_at'] = df['created_at'].dt.tz_localize(None)
for col, name in [('field1','suhu'), ('field2','udara'), ('field3','tanah')]:
    df[name] = pd.to_numeric(df[col], errors='coerce')
df = df[['created_at','suhu','udara','tanah']].set_index('created_at')

RANGES = {'suhu': (10, 45), 'udara': (0, 100), 'tanah': (0, 100)}
for v,(lo,hi) in RANGES.items():
    df[v] = df[v].where((df[v]>=lo)&(df[v]<=hi))
for v in RANGES:
    s = df[v]; med = s.median(); mad = (s-med).abs().median()
    if mad and mad>0: df[v] = s.where((s-med).abs() <= 3.5*1.4826*mad)

harian = df.resample('D').mean().interpolate(method='time', limit=3, limit_area='inside').dropna(how='all')
print('Jumlah hari:', len(harian))
harian.tail()

## 2. Windowing (deret waktu → data terawasi)

Variabel fokus bisa diganti (`VAR`). Window = berapa hari ke belakang dipakai untuk memprediksi 1 hari berikut.

In [ ]:
VAR = 'suhu'   # ganti ke 'udara' / 'tanah' bila mau
W = 7          # ukuran window (hari)
K = 10         # jumlah hari untuk DATA UJI (test)

s = harian[VAR].dropna()
dates = s.index
vals = s.values.astype('float32')

X, y, ydates = [], [], []
for i in range(len(vals) - W):
    X.append(vals[i:i+W]); y.append(vals[i+W]); ydates.append(dates[i+W])
X = np.array(X); y = np.array(y)

Xtr, Xte = X[:-K], X[-K:]
ytr, yte = y[:-K], y[-K:]
ydates_te = ydates[-K:]

# Normalisasi (fit di TRAIN saja -> hindari kebocoran data)
mn, mx = float(Xtr.min()), float(Xtr.max())
norm   = lambda a: (a - mn) / (mx - mn + 1e-9)
denorm = lambda a: a * (mx - mn + 1e-9) + mn
Xtr_n, Xte_n = norm(Xtr), norm(Xte)
ytr_n = norm(ytr)

print(f'Variabel={VAR} | sampel={len(X)} (train={len(Xtr)}, test={len(Xte)}) | window={W}')

## 3. Metrik

In [ ]:
def mae(a,p):  a,p=np.asarray(a,float),np.asarray(p,float); return float(np.mean(np.abs(a-p)))
def rmse(a,p): a,p=np.asarray(a,float),np.asarray(p,float); return float(np.sqrt(np.mean((a-p)**2)))
def mape(a,p):
    a,p=np.asarray(a,float),np.asarray(p,float); m=np.abs(a)>1e-6
    return float(np.mean(np.abs((a[m]-p[m])/a[m]))*100) if m.sum() else float('nan')
def akurasi(a,p): return float(max(0.0, 100-mape(a,p)))

## 4. Latih MLP (ANN), DNN, LSTM

- **MLP (ANN)**: 1 hidden layer — jaringan saraf paling dasar.
- **DNN**: beberapa hidden layer + Dropout (lebih "dalam").
- **LSTM**: jaringan untuk urutan/deret waktu.

EarlyStopping mencegah overfitting. Riwayat training disimpan untuk kurva loss/MAE.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
tf.random.set_seed(42); np.random.seed(42)

def latih(model, reshape=False):
    Xt = Xtr_n.reshape(-1, W, 1) if reshape else Xtr_n
    es = callbacks.EarlyStopping(patience=40, restore_best_weights=True)
    return model.fit(Xt, ytr_n, validation_split=0.2, epochs=400, batch_size=8, verbose=0, callbacks=[es])

# MLP (ANN) — 1 hidden layer
mlp = models.Sequential([layers.Input((W,)), layers.Dense(16, activation='relu'), layers.Dense(1)])
mlp.compile(optimizer='adam', loss='mse', metrics=['mae'])
h_mlp = latih(mlp)

# DNN — lebih dalam + dropout
dnn = models.Sequential([layers.Input((W,)),
                         layers.Dense(32, activation='relu'), layers.Dropout(0.2),
                         layers.Dense(16, activation='relu'), layers.Dense(1)])
dnn.compile(optimizer='adam', loss='mse', metrics=['mae'])
h_dnn = latih(dnn)

# LSTM — input urutan (samples, window, 1)
lstm = models.Sequential([layers.Input((W,1)), layers.LSTM(16), layers.Dense(1)])
lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
h_lstm = latih(lstm, reshape=True)

print('Selesai latih. Epoch terpakai:',
      'MLP', len(h_mlp.history['loss']), '| DNN', len(h_dnn.history['loss']), '| LSTM', len(h_lstm.history['loss']))

## 5. Kurva Loss & MAE per epoch (otentik pada neural network)

Loss (MSE) idealnya **turun**. Bila `val_loss` naik sementara `loss` turun -> tanda **overfitting**.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(12, 11))
for row,(h,nama) in enumerate([(h_mlp,'MLP (ANN)'), (h_dnn,'DNN'), (h_lstm,'LSTM')]):
    axes[row,0].plot(h.history['loss'], label='train'); axes[row,0].plot(h.history['val_loss'], label='val')
    axes[row,0].set_title(nama+' - Loss (MSE)'); axes[row,0].set_xlabel('epoch'); axes[row,0].grid(alpha=.3); axes[row,0].legend()
    axes[row,1].plot(h.history['mae'], label='train'); axes[row,1].plot(h.history['val_mae'], label='val')
    axes[row,1].set_title(nama+' - MAE (error, makin kecil makin baik)'); axes[row,1].set_xlabel('epoch'); axes[row,1].grid(alpha=.3); axes[row,1].legend()
plt.tight_layout(); plt.show()

## 6. Bandingkan SEMUA model di data uji (MAE / RMSE / Akurasi)

Semua dievaluasi pada data uji yang SAMA: Naive, Linear (regresi pada lag), Prophet, MLP, DNN, LSTM.

In [ ]:
# Prediksi DL (lalu kembalikan ke skala asli)
def pred_dl(model, reshape=False):
    Xt = Xte_n.reshape(-1, W, 1) if reshape else Xte_n
    return denorm(model.predict(Xt, verbose=0).flatten())
p_mlp, p_dnn, p_lstm = pred_dl(mlp), pred_dl(dnn), pred_dl(lstm, True)

# Naive: nilai terakhir di window
p_naive = Xte[:, -1]

# Linear: regresi linear pada fitur lag
from sklearn.linear_model import LinearRegression
p_lin = LinearRegression().fit(Xtr, ytr).predict(Xte)

# Prophet: dilatih pada data SEBELUM tanggal uji
import logging
logging.getLogger('prophet').setLevel(logging.WARNING); logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
from prophet import Prophet
dfp = harian[[VAR]].dropna().reset_index(); dfp.columns=['ds','y']
if dfp['ds'].dt.tz is not None: dfp['ds']=dfp['ds'].dt.tz_localize(None)
mP = Prophet(weekly_seasonality=False, daily_seasonality=False, yearly_seasonality=False, changepoint_prior_scale=0.01)
mP.fit(dfp[dfp['ds'] < ydates_te[0]])
p_prophet = mP.predict(pd.DataFrame({'ds': list(ydates_te)}))['yhat'].values

hasil = []
for nama, pred in [('Naive',p_naive), ('Linear',p_lin), ('Prophet',p_prophet),
                   ('MLP (ANN)',p_mlp), ('DNN',p_dnn), ('LSTM',p_lstm)]:
    hasil.append({'model':nama, 'MAE':round(mae(yte,pred),3),
                  'RMSE':round(rmse(yte,pred),3), 'Akurasi(%)':round(akurasi(yte,pred),1)})
tabel = pd.DataFrame(hasil).sort_values('MAE').reset_index(drop=True)
print('Variabel:', VAR)
print(tabel.to_string(index=False))
tabel

## 7. Grafik perbandingan

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
tt = tabel.set_index('model')
tt[['MAE','RMSE']].plot(kind='bar', ax=ax1, color=['#ef4444','#9ca3af'])
ax1.set_title('Error (makin kecil makin baik) - '+VAR); ax1.set_ylabel('error'); ax1.grid(alpha=.3, axis='y'); ax1.tick_params(axis='x', rotation=30)
tt['Akurasi(%)'].plot(kind='bar', ax=ax2, color='#16a34a'); ax2.set_title('Akurasi (100 - MAPE)'); ax2.set_ylim(0,100); ax2.grid(alpha=.3, axis='y'); ax2.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 8. Aktual vs Prediksi (data uji)

In [ ]:
plt.figure(figsize=(11,4))
plt.plot(ydates_te, yte, 'o-', color='#111', label='Aktual', linewidth=2)
for nama, pred, c in [('Prophet',p_prophet,'#16a34a'), ('MLP',p_mlp,'#ef4444'), ('DNN',p_dnn,'#f59e0b'), ('LSTM',p_lstm,'#3b82f6')]:
    plt.plot(ydates_te, pred, '--', color=c, label=nama, alpha=.8)
plt.title('Aktual vs Prediksi - '+VAR); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 9. Kesimpulan (untuk laporan)

- Bandingkan tabel: sering **model sederhana (Naive/Linear/Prophet) menang** pada data sedikit — bukti *bias-variance & data hunger* deep learning.
- **Kurva Loss/MAE** menunjukkan proses belajar neural network (dan tanda overfitting bila val naik).
- Tangga pembelajaran: Naive -> Linear -> Prophet -> **ANN (MLP)** -> **DNN** -> **LSTM**.
- Untuk menaikkan performa DL: tambah data harian secara rutin, lalu latih ulang.

> Ingin model DL ini tampil di website? Simpan prediksinya (sama seperti Prophet) ke model_config.json,
> atau jalankan ANN langsung di browser dengan TensorFlow.js.